# 第 46 课：Qwen3-ASR——真实前沿模型的推理、微调与验收

这一课把前 45 课连接到真实开放模型。你将完成硬件预检、官方数据格式、speaker-disjoint 切分、CER/WER、可选真实推理，以及一套不会把训练集过拟合冒充效果的微调流程。

当前项目环境是 CPU，因此所有数据与评测实验都可运行；模型下载和 GPU 推理默认关闭，避免意外下载数 GB 权重。

## 完成标准

1. 能画出 Qwen3-ASR 的 AuT encoder → projector → Qwen3；
2. 能选择 0.6B/1.7B、offline/streaming 后端；
3. 生成并验证官方 JSONL 格式，且 train/eval 说话人不重叠；
4. 从空白实现 CER，并用手算样例验收；
5. 写出 baseline → SFT → 独立评测 → 部署门禁的完整方案；
6. 有 GPU 时能启用可选单元完成真实转录。

## 1. 2026 架构快照

根据 Qwen3-ASR Technical Report：

```text
16 kHz 音频
→ 128 维 Fbank
→ AuT encoder
→ 8 倍下采样，约 12.5 Hz speech embedding
→ Projector
→ Qwen3-0.6B / Qwen3-1.7B
→ 自回归 ASR 文本
```

- 0.6B 版本：180M AuT encoder，hidden size 896，强调准确率—效率平衡；
- 1.7B 版本：300M AuT encoder，hidden size 1024，强调更高质量；
- 动态注意力窗 1～8 秒，用一个模型兼容 offline/streaming；
- 官方共支持 52 种语言和方言（30 种语言、22 种中文方言）；
- 训练经历 AuT 预训练、Qwen3-Omni 预训练、ASR SFT 与 GSPO RL。

资料：

- 技术报告：https://arxiv.org/abs/2601.21337
- 官方仓库：https://github.com/QwenLM/Qwen3-ASR
- 官方微调说明：https://github.com/QwenLM/Qwen3-ASR/tree/main/finetuning

厂商报告中的 SOTA、速度和内部集结果是选型线索，不是你自己场景的验收结论。

## 2. 硬件与后端预检

Transformers 后端适合先验证离线推理；官方仓库当前的流式推理使用 vLLM 后端。vLLM/FlashAttention 通常要求合适的 NVIDIA CUDA 环境，Windows 用户更适合 WSL2/Linux 或云 GPU。

In [1]:
import importlib.util
import json
import math
import platform
import re
import string
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

ROOT = Path.cwd()
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("qwen_asr installed:", importlib.util.find_spec("qwen_asr") is not None)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("当前走 CPU 教学路径；真实 Qwen3-ASR 推理单元保持关闭。")

python: 3.13.12
platform: Windows-11-10.0.26200-SP0
torch: 2.13.0+cpu
CUDA available: False
qwen_asr installed: False
当前走 CPU 教学路径；真实 Qwen3-ASR 推理单元保持关闭。


In [2]:
def prefix_budget(audio_seconds, output_hz=12.5):
    return math.ceil(audio_seconds * output_hz)


for seconds in [1, 10, 60, 1200]:
    print(f"{seconds:4d} 秒 → 约 {prefix_budget(seconds):5d} 个 AuT 输出时间步")

assert prefix_budget(10) == 125
print("断言通过：下采样降低序列长度，但 20 分钟仍是很长的注意力输入。")

   1 秒 → 约    13 个 AuT 输出时间步
  10 秒 → 约   125 个 AuT 输出时间步
  60 秒 → 约   750 个 AuT 输出时间步
1200 秒 → 约 15000 个 AuT 输出时间步
断言通过：下采样降低序列长度，但 20 分钟仍是很长的注意力输入。


## 3. 建立不会泄漏的数据集

官方 SFT JSONL 每行至少包含：

```json
{"audio":"/path/utt.wav","text":"language Chinese<asr_text>你好世界"}
```

有语言标签时使用 `language Chinese/English...`；没有时使用 `language None`。但工业数据还应在自己的元数据表保存 `speaker_id、device、domain、duration、license`，用于分组切分和分桶评测。

下面使用仓库内 FSDD 多说话人样例。切分单位必须是 speaker，而不是随机切音频，否则同一个人的音色会同时出现在训练和评估中。

In [3]:
DIGIT_TEXT = {0: "zero", 1: "one", 2: "two"}
audio_files = sorted((ROOT / "data" / "fsdd_multispeaker").glob("*.wav"))
records = []
for path in audio_files:
    match = re.fullmatch(r"(\d+)_([^_]+)_\d+\.wav", path.name)
    if not match:
        continue
    digit = int(match.group(1))
    speaker = match.group(2)
    info = sf.info(path)
    records.append({
        "audio": str(path.resolve()),
        "text": f"language English<asr_text>{DIGIT_TEXT[digit]}",
        "plain_text": DIGIT_TEXT[digit],
        "speaker_id": speaker,
        "sample_rate": info.samplerate,
        "duration": info.frames / info.samplerate,
    })

speakers = sorted({r["speaker_id"] for r in records})
eval_speaker = speakers[-1]
train_records = [r for r in records if r["speaker_id"] != eval_speaker]
eval_records = [r for r in records if r["speaker_id"] == eval_speaker]
train_speakers = {r["speaker_id"] for r in train_records}
eval_speakers = {r["speaker_id"] for r in eval_records}

print("speakers:", speakers)
print("train/eval records:", len(train_records), len(eval_records))
print("train speakers:", train_speakers, "eval speakers:", eval_speakers)
print("官方 JSONL 行示例:")
print(json.dumps({k: train_records[0][k] for k in ["audio", "text"]}, ensure_ascii=False))

assert records and train_speakers.isdisjoint(eval_speakers)
assert all(Path(r["audio"]).is_file() for r in records)
assert all(r["sample_rate"] > 0 and r["duration"] > 0 for r in records)
print("断言通过：路径、音频属性有效，train/eval 说话人完全隔离。")

speakers: ['george', 'nicolas', 'theo', 'yweweler']
train/eval records: 9 3
train speakers: {'george', 'theo', 'nicolas'} eval speakers: {'yweweler'}
官方 JSONL 行示例:
{"audio": "G:\\learn_asr\\data\\fsdd_multispeaker\\0_george_0.wav", "text": "language English<asr_text>zero"}
断言通过：路径、音频属性有效，train/eval 说话人完全隔离。


In [4]:
# 改成 True 才会写文件；已运行版本不会修改你的数据目录。
WRITE_MANIFESTS = False
if WRITE_MANIFESTS:
    manifest_dir = ROOT / "data" / "qwen3_asr_manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)
    for split, items in [("train", train_records), ("eval", eval_records)]:
        output = manifest_dir / f"{split}.jsonl"
        with output.open("w", encoding="utf-8") as handle:
            for item in items:
                official = {"audio": item["audio"], "text": item["text"]}
                handle.write(json.dumps(official, ensure_ascii=False) + "\n")
        print("wrote", output)
else:
    print("WRITE_MANIFESTS=False：只验证，不写文件。")

WRITE_MANIFESTS=False：只验证，不写文件。


## 4. CER/WER：先把尺子做对

中文通常报告 CER，空格分词语言通常报告 WER：

$$\mathrm{ErrorRate}=\frac{S+D+I}{N}$$

`S/D/I` 分别是替换、删除、插入，`N` 是参考序列长度。文本规范化必须在实验开始前固定并版本化；随结果修改规则会污染比较。

In [5]:
def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, ref_item in enumerate(reference, start=1):
        current = [i]
        for j, hyp_item in enumerate(hypothesis, start=1):
            substitute = previous[j - 1] + (ref_item != hyp_item)
            delete = previous[j] + 1
            insert = current[j - 1] + 1
            current.append(min(substitute, delete, insert))
        previous = current
    return previous[-1]


def normalize_zh(text):
    # 教学规则：保留中文、字母和数字；生产规则必须覆盖数字/金额/日期等。
    return "".join(ch.lower() for ch in text if ch.isalnum())


def normalize_en(text):
    table = str.maketrans("", "", string.punctuation)
    return text.lower().translate(table).split()


def cer(reference, hypothesis):
    ref = list(normalize_zh(reference))
    hyp = list(normalize_zh(hypothesis))
    return edit_distance(ref, hyp) / max(1, len(ref))


def wer(reference, hypothesis):
    ref = normalize_en(reference)
    hyp = normalize_en(hypothesis)
    return edit_distance(ref, hyp) / max(1, len(ref))


print("CER:", cer("今天天气很好", "今天天气好"))
print("WER:", wer("we learn speech models", "we learn models"))
assert cer("今天天气很好", "今天天气好") == 1 / 6
assert wer("we learn speech models", "we learn models") == 1 / 4
assert cer("完全相同", "完全相同") == 0
print("断言通过：手算删除错误与实现一致。")

CER: 0.16666666666666666
WER: 0.25
断言通过：手算删除错误与实现一致。


## 5. 可选：真实模型推理

建议为真实模型单独建立 Python 3.12 + CUDA 环境，不要破坏本课程 CPU 环境：

```powershell
uv venv .venv-qwen --python 3.12
.\.venv-qwen\Scripts\Activate.ps1
uv pip install -U qwen-asr
```

流式/vLLM 路线按照官方仓库安装 `qwen-asr[vllm]`，通常在 Linux/WSL2 或云 GPU 上进行。先用 0.6B 验证资源与数据契约，再决定是否换 1.7B。

下面的开关默认为 `False`。开启前确认已有 CUDA、足够显存/内存、模型下载许可与磁盘空间。

In [6]:
RUN_REAL_MODEL = False
MODEL_ID = "Qwen/Qwen3-ASR-0.6B"

if RUN_REAL_MODEL:
    if not torch.cuda.is_available():
        raise RuntimeError("真实模型实验需要可用 CUDA；请切换到 GPU 环境。")
    if importlib.util.find_spec("qwen_asr") is None:
        raise RuntimeError("请先在独立环境安装：uv pip install -U qwen-asr")

    from qwen_asr import Qwen3ASRModel

    real_model = Qwen3ASRModel.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        device_map="cuda:0",
        max_inference_batch_size=4,
        max_new_tokens=128,
    )
    sample_path = eval_records[0]["audio"]
    started = time.perf_counter()
    result = real_model.transcribe(audio=sample_path, language="English")[0]
    elapsed = time.perf_counter() - started
    duration = eval_records[0]["duration"]
    print("reference:", eval_records[0]["plain_text"])
    print("language:", result.language)
    print("prediction:", result.text)
    print(f"elapsed={elapsed:.3f}s RTF={elapsed/duration:.3f}")
else:
    print("RUN_REAL_MODEL=False：未下载权重，CPU 课程执行保持轻量。")

RUN_REAL_MODEL=False：未下载权重，CPU 课程执行保持轻量。


## 6. 官方 SFT 命令与正确实验顺序

官方微调脚本支持 JSONL 音频—文本对、单 GPU 和 `torchrun` 多 GPU。示例核心参数为：

```powershell
python qwen3_asr_sft.py `
  --model_path Qwen/Qwen3-ASR-0.6B `
  --train_file .\train.jsonl `
  --eval_file .\eval.jsonl `
  --output_dir .\qwen3-asr-sft-out `
  --batch_size 4 `
  --grad_acc 8 `
  --lr 2e-5 `
  --epochs 1 `
  --save_steps 200
```

课程将 batch 调小只是为了降低显存门槛，不代表最佳超参数。正确顺序是：

1. **冻结数据与规范**：许可、去重、speaker/domain split、文本规范版本；
2. **跑零样本 baseline**：保存逐条输出、CER/WER、RTF 和失败桶；
3. **小批量过拟合**：先证明数据格式、loss、checkpoint 和加载链路正确；
4. **正式 SFT**：只根据 validation 选 checkpoint，绝不看 test 调参；
5. **独立 test**：总体与方言/噪声/设备/长度/专名/数字分桶；
6. **回归门禁**：新领域提升不能用通用能力、静音幻觉或延迟恶化换取；
7. **部署验收**：并发、显存、首 token、尾延迟、崩溃恢复和版本回滚。

## 7. 一张最低验收表

| 维度 | 必测项 |
|---|---|
| 质量 | CER/WER、专名召回、数字准确率、漏段率 |
| 鲁棒性 | 静音、音乐、低 SNR、重叠说话、口音/方言、儿童/老人 |
| 忠实度 | 否定词、数字、姓名、提示注入、无语音幻觉 |
| 流式 | 首字延迟、partial 修改次数、endpoint 尾延迟 |
| 性能 | RTF、并发吞吐、p50/p95/p99、峰值显存 |
| 工程 | 模型/词表/规范版本、回滚、日志脱敏、授权与许可 |

“平均 CER 降低”不足以发布。例如金额识别恶化、静音产生文本或 p99 延迟翻倍，都可能阻止上线。

## 分层练习（24 分）

### A. 回忆（每题 1 分）

1. Qwen3-ASR 三个主要结构部件是什么？
2. 0.6B 和 1.7B 应怎样做第一次选型？
3. 官方 JSONL 的两个必需字段是什么？
4. CER 分母是什么？

### B. 推理（每题 2 分）

5. 60 秒音频在 12.5 Hz 下约有多少 speech embedding？
6. 为什么随机按 utterance 切分可能造成 speaker 泄漏？
7. 微调后 validation CER 降、test CER 升，可能是什么原因？
8. 为什么必须单独测试静音和否定词？

### C. 实战（每题 3 分）

9. 为自己的 10 条音频生成 JSONL，并通过路径/采样率检查。
10. 从空白实现 edit distance，用三组手算例子验证。
11. 有 GPU 时运行 0.6B baseline，保存逐条预测与 RTF。
12. 写出包含数据、质量、延迟、幻觉和回滚的上线门禁。

达到 19/24 只是理论通关；真正完成还需在你自己的、从未参与训练的数据上跑出可复现报告。

## 结业综合任务

选择一个真实场景（会议、客服、短视频、车载或方言），提交：

1. 需求：语言、是否流式、延迟与设备预算；
2. 数据卡：来源、许可、说话人切分、时长、噪声和文本规范；
3. baseline：至少一个 CTC/RNN-T 系统与一个 LALM；
4. 指标：总体与分桶 CER/WER、RTF、延迟、幻觉测试；
5. 搭建图：前端、模型、解码、后处理、服务和状态；
6. 失败分析：至少 20 条错误分类；
7. 改进实验：一次只改变一个变量；
8. 复现说明：代码、配置、随机种子、模型和数据版本。

能独立完成并答辩这八项，才算从“会运行模型”进入“会搭建和评估 ASR 系统”。

## 离场小测（闭卷发给老师）

1. 画出 Qwen3-ASR 数据流并标注下采样。
2. 为什么一定要先跑 baseline 再微调？
3. 给出 CER 公式，并手算一个例子。
4. 设计三条能发现生成式 ASR 幻觉的音频。
5. 你的目标场景是什么？质量、延迟、硬件三项约束分别是什么？

附上本课断言结果；有 GPU 时再附真实 baseline 表。